# Johansen CO₂ Storage — Injection Rate Sensitivity Study
## Comprehensive Exploratory Data Analysis

**5 simulation runs at different CO₂ injection rates (3.5 → 56 Mt/yr, doubling each run)**

**Sections:**
1. Data Loading & Catalogue
2. Geographic Map — Click a Well to See Its Full Data Across All Scenarios
3. Injection Rate Sensitivity — Injector Pressure & Saturation
4. Pressure Propagation Across Observation Wells
5. Formation Capacity Analysis (Peak Pressure vs Rate)
6. Scenario × Well Heatmap
7. Pressure Recovery Time Analysis
8. Interactive Multi-Well Explorer

In [ ]:
import os, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output
warnings.filterwarnings('ignore')

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'well_csvs'))
DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'data'))
if not os.path.exists(BASE_DIR):
    BASE_DIR = '/Users/apple/Desktop/study/programming/Matlab/Plugins/MRST-2026a/core/examples/data/Johansen/well_csvs'
    DATA_DIR = '/Users/apple/Desktop/study/programming/Matlab/Plugins/MRST-2026a/core/examples/data/Johansen/data'
WELL_LOC_FILE = os.path.join(DATA_DIR, 'well_loc.csv')

print(f'Base CSVs : {BASE_DIR}')
print(f'Data dir  : {DATA_DIR}')

In [ ]:
# ── 1. Data Loading & Catalogue ────────────────────────────────────────────

def parse_summary(path):
    result = {'active_wells': {}, 'obs_wells': {}, 'meta': {}}
    if not os.path.exists(path):
        return result
    with open(path) as f:
        lines = f.readlines()
    mode = None
    for line in lines:
        s = line.strip()
        for key, tag in [('co2_total','Total CO2 injected'), ('brine_total','Total brine produced'),
                         ('peak_bhp','Peak injector BHP'), ('sim_window','Simulation window'),
                         ('inj_end','Injection end year')]:
            if tag in s:
                result['meta'][key] = s.split(':')[-1].strip()
        if '--- ACTIVE PLAN WELLS' in s:   mode = 'plan'; continue
        if '--- MASS BALANCE' in s:         mode = None;   continue
        if '--- OBSERVATION WELLS' in s:    mode = 'obs';  continue
        if '--- CSV COLUMN' in s:           mode = None;   continue
        if mode == 'plan':
            if not s or s.startswith('Well') or s.startswith('---'): continue
            role = 'Injector' if 'Injector' in s else ('Producer' if 'Producer' in s else None)
            if role:
                wname = s.split(role)[0].strip()
                try:    rate_val = float(s.split('Mt/yr')[0].split()[-1])
                except: rate_val = 0.0
                result['active_wells'][wname] = {'role': role, 'rate_mtyr': rate_val}
        elif mode == 'obs':
            if not s or s.startswith('Well') or s.startswith('---'): continue
            parts = s.split()
            if len(parts) >= 7:
                wname = ' '.join(parts[:-6])
                result['obs_wells'][wname] = {
                    'GridI': int(parts[-6]), 'GridJ': int(parts[-5]),
                    'depth_m': float(parts[-2]), 'csv': parts[-1]
                }
    return result

run_folders = sorted([f for f in os.listdir(BASE_DIR) if os.path.isdir(os.path.join(BASE_DIR, f))])
runs = {}
for rf in run_folders:
    rfpath  = os.path.join(BASE_DIR, rf)
    summary = parse_summary(os.path.join(rfpath, 'simulation_summary.txt'))
    rate_val   = 0.0
    rate_label = 'Unknown'
    for aw in summary['active_wells'].values():
        if aw['role'] == 'Injector' and aw['rate_mtyr'] > 0:
            rate_val   = aw['rate_mtyr']
            rate_label = f"{aw['rate_mtyr']:.1f} Mt/yr"
    well_dfs = {}
    for wname, winfo in summary['obs_wells'].items():
        csv_path = os.path.join(rfpath, winfo['csv'])
        if os.path.exists(csv_path):
            df = pd.read_csv(csv_path)
            df['Well']       = wname
            df['Run']        = rf
            df['Rate_label'] = rate_label
            well_dfs[wname]  = df
    runs[rf] = {
        'label': rate_label, 'rate_val': rate_val,
        'meta': summary['meta'],
        'active_wells': summary['active_wells'],
        'obs_wells':    summary['obs_wells'],
        'dfs':          well_dfs
    }

all_dfs = pd.concat(
    [df for rd in runs.values() for df in rd['dfs'].values()],
    ignore_index=True
)

print(f'Loaded {len(run_folders)} simulation runs  |  Total records: {len(all_dfs):,}')
print()
print(f'  {"Run Folder":<28} {"Inj Rate":>12}  {"Total CO2":>18}  {"Peak BHP":>14}')
print('  ' + '-'*78)
for rf, rd in runs.items():
    m = rd['meta']
    print(f'  {rf:<28} {rd["label"]:>12}  {m.get("co2_total","N/A"):>18}  {m.get("peak_bhp","N/A"):>14}')

In [ ]:
# ── 2. Geographic Map — Click a Well to Explore Its Data ──────────────────
#
#  Rules applied:
#    • All out-of-bounds wells removed entirely.
#    • Exact duplicate lat/lon collapsed to ONE representative well.
#      Priority: Active Injector > Active Producer > Obs (In-Reservoir).
#    • Clicking any well plots its Pressure + CO₂ Saturation across
#      ALL 5 scenarios in the output cell below the map.

# Load well locations
df_loc = pd.read_csv(WELL_LOC_FILE)
refLat, refLon, refI, refJ = 60.576425, 3.443367, 51, 51
df_loc['GridI']   = (refI + (df_loc['EW_DEC_DEG'] - refLon) * (55.5 / 0.9)).round().astype(int)
df_loc['GridJ']   = (refJ + (df_loc['NS_DEC_DEG'] - refLat) * (111.0 / 0.9)).round().astype(int)
df_loc['InBounds'] = df_loc['GridI'].between(1,100) & df_loc['GridJ'].between(1,100)

# Latest run for status reference
latest_run = list(runs.keys())[-1]
latest     = runs[latest_run]

def get_status(wname):
    if wname in latest['active_wells']:
        return 'Active ' + latest['active_wells'][wname]['role']
    if wname in latest['obs_wells']:
        return 'In-Reservoir (Obs)'
    return 'Out-of-Bounds'

# Status priority score (lower = higher priority)
STATUS_PRIORITY = {'Active Injector': 0, 'Active Producer': 1, 'In-Reservoir (Obs)': 2, 'Out-of-Bounds': 99}

df_loc['Status']   = df_loc['Well_Bore_Name'].map(get_status)
df_loc['Priority'] = df_loc['Status'].map(STATUS_PRIORITY)

# Step 1: Remove all out-of-bounds wells
df_in = df_loc[df_loc['InBounds']].copy()

# Step 2: Deduplicate exact lat/lon — keep highest-priority well per location
df_in = df_in.sort_values('Priority')  # lowest priority number first = highest importance
df_dedup = df_in.drop_duplicates(subset=['NS_DEC_DEG', 'EW_DEC_DEG'], keep='first').copy()

# Build a mapping: representative well name → list of all coincident names at that location
loc_to_all = {}
for (lat, lon), grp in df_in.groupby(['NS_DEC_DEG', 'EW_DEC_DEG']):
    all_names = grp['Well_Bore_Name'].tolist()
    rep_name  = df_dedup.loc[(df_dedup['NS_DEC_DEG']==lat) & (df_dedup['EW_DEC_DEG']==lon), 'Well_Bore_Name'].values[0]
    loc_to_all[rep_name] = all_names

print(f'In-bounds wells: {len(df_in)}  →  After deduplication: {len(df_dedup)} unique locations')
print('Collapsed groups:')
for rep, all_names in loc_to_all.items():
    if len(all_names) > 1:
        others = [n for n in all_names if n != rep]
        print(f'  Representative: {rep:20s}  |  Collapsed: {others}')

# ── Build FigureWidget (supports click callbacks) ──────────────────────────
style = {
    'Active Injector':    {'color': '#d62728', 'symbol': 'triangle-up', 'size': 16},
    'Active Producer':    {'color': '#1f77b4', 'symbol': 'square',       'size': 16},
    'In-Reservoir (Obs)': {'color': '#2ca02c', 'symbol': 'circle',       'size': 11},
}

fig_map = go.FigureWidget()

for status, s in style.items():
    sub = df_dedup[df_dedup['Status'] == status]
    if sub.empty: continue
    hover_texts = []
    custom_data = []
    for _, row in sub.iterrows():
        wname = row['Well_Bore_Name']
        all_at_loc = loc_to_all.get(wname, [wname])
        others = [n for n in all_at_loc if n != wname]
        coinc_str = f'<br>⚠️ Also at this location: {", ".join(others)}' if others else ''
        hover_texts.append(
            f'<b>{wname}</b><br>'
            f'Status: <b>{status}</b><br>'
            f'Grid: ({row["GridI"]}, {row["GridJ"]})<br>'
            f'Lat {row["NS_DEC_DEG"]:.5f}°N  Lon {row["EW_DEC_DEG"]:.5f}°E'
            f'{coinc_str}<br>'
            f'<i>Click to see time-series data</i>'
        )
        custom_data.append([wname])
    fig_map.add_trace(go.Scatter(
        x=sub['EW_DEC_DEG'].tolist(),
        y=sub['NS_DEC_DEG'].tolist(),
        mode='markers+text',
        name=status,
        text=sub['Well_Bore_Name'].tolist(),
        textposition='top center',
        hovertext=hover_texts,
        hoverinfo='text',
        customdata=custom_data,
        marker=dict(size=s['size'], color=s['color'], symbol=s['symbol'],
                    line=dict(width=1.5, color='black'))
    ))

fig_map.update_layout(
    title='<b>Johansen Formation — In-Reservoir Wells</b><br>'
          '(Out-of-bounds & exact duplicates removed. Click any well for time-series data.)',
    xaxis_title='Longitude (°E)', yaxis_title='Latitude (°N)',
    template='plotly_white', height=580,
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.9)', borderwidth=1),
    margin=dict(t=80, b=40)
)

# ── Click Handler ─────────────────────────────────────────────────────────
click_output = widgets.Output()

def on_well_click(trace, points, selector):
    if not points.point_inds:
        return
    idx       = points.point_inds[0]
    well_name = trace.customdata[idx][0]

    with click_output:
        clear_output(wait=True)

        # Find data for this well across all runs
        found_runs = []
        for rf, rd in runs.items():
            if well_name in rd['dfs']:
                found_runs.append((rf, rd, well_name))
            else:
                # Check if any coincident alias has data
                aliases = loc_to_all.get(well_name, [well_name])
                for alias in aliases:
                    if alias in rd['dfs']:
                        found_runs.append((rf, rd, alias))
                        break

        if not found_runs:
            print(f'No simulation data found for well: {well_name}')
            return

        n_runs  = len(found_runs)
        palette = cm.plasma(np.linspace(0.1, 0.9, n_runs))

        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
        fig.patch.set_facecolor('#ffffff')

        winfo_depth = 'N/A'
        for i, (rf, rd, dname) in enumerate(found_runs):
            df    = rd['dfs'][dname]
            label = rd['label']  # e.g. '3.5 Mt/yr'
            color = palette[i]

            if winfo_depth == 'N/A' and 'Mean_Depth_m' in df.columns:
                winfo_depth = f"{df['Mean_Depth_m'].iloc[0]:.1f} m"

            ax1.plot(df['Time_yr'], df['Pressure_bar'],
                     color=color, linewidth=2, label=label)
            ax2.plot(df['Time_yr'], df['CO2_Saturation'],
                     color=color, linewidth=2, label=label)

        # Injection end marker (use first run)
        try:
            inj_end = float(list(runs.values())[0]['meta'].get('inj_end', '50').split()[0])
            ax1.axvline(inj_end, color='gray', ls=':', lw=1.5, label=f'Inj. end (Yr {inj_end:.0f})')
            ax2.axvline(inj_end, color='gray', ls=':', lw=1.5)
        except: pass

        # Status & depth info
        status_str = get_status(well_name)
        aliases    = loc_to_all.get(well_name, [well_name])
        coinc_str  = f'  |  Also at location: {", ".join([a for a in aliases if a != well_name])}' if len(aliases)>1 else ''

        # Auto-scale pressure y-axis
        all_p = pd.concat([rd['dfs'].get(dname, pd.DataFrame({'Pressure_bar': []}))['Pressure_bar']
                           for _, rd, dname in found_runs])
        if not all_p.empty:
            pmin, pmax = all_p.min(), all_p.max()
            margin = max(2, (pmax - pmin) * 0.05)
            ax1.set_ylim(pmin - margin, pmax + margin)

        ax1.set_ylabel('Pressure (bar)', fontsize=11, fontweight='bold')
        ax1.grid(True, ls='--', alpha=0.5)
        ax1.legend(title='Injection Rate', fontsize=9, loc='upper right', ncol=2)
        ax1.set_title(
            f'Reservoir Pressure vs Time — {well_name}\n'
            f'Status: {status_str}  |  Depth: {winfo_depth}{coinc_str}',
            fontsize=11, fontweight='bold', pad=8
        )

        ax2.set_ylabel('CO₂ Saturation (fraction)', fontsize=11, fontweight='bold')
        ax2.set_xlabel('Time (years)', fontsize=11)
        ax2.grid(True, ls='--', alpha=0.5)
        ax2.legend(title='Injection Rate', fontsize=9, loc='upper right', ncol=2)
        ax2.set_title(f'CO₂ Saturation vs Time — {well_name}', fontsize=11, fontweight='bold', pad=8)

        plt.suptitle(
            f'All Scenarios Compared | Well: {well_name} | '
            f'Runs: {n_runs} ({" → ".join([rd["label"] for _, rd, _ in found_runs])})',
            fontsize=11, y=1.01
        )
        plt.tight_layout()
        plt.show()

        # Print summary table
        print(f'  Peak Values Across Scenarios — {well_name}')
        print(f'  {"-"*65}')
        print(f'  {"Scenario":<28} {"Rate":>12}  {"Peak P (bar)":>14}  {"Peak CO₂ Sat":>14}')
        print(f'  {"-"*65}')
        for _, rd, dname in found_runs:
            df = rd['dfs'][dname]
            print(f'  {_:<28} {rd["label"]:>12}  {df["Pressure_bar"].max():>14.2f}  {df["CO2_Saturation"].max():>14.4f}')

for trace in fig_map.data:
    trace.on_click(on_well_click)

display(widgets.VBox([
    fig_map,
    widgets.HTML('<hr><b>Well Time-Series (click any well above to populate):</b>'),
    click_output
]))

In [ ]:
# ── 3. Injection Rate Sensitivity — Injector Well ─────────────────────────
INJ_WELL = '31/05/07'
palette  = cm.plasma(np.linspace(0.1, 0.9, len(runs)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
for ax, col, ylabel in zip(axes, ['Pressure_bar','CO2_Saturation'], ['Pressure (bar)','CO₂ Saturation']):
    for i, (rf, rd) in enumerate(runs.items()):
        if INJ_WELL in rd['dfs']:
            df = rd['dfs'][INJ_WELL]
            ax.plot(df['Time_yr'], df[col], color=palette[i], linewidth=2, label=rd['label'])
    ax.set_xlabel('Time (years)', fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.grid(True, ls='--', alpha=0.5)
    ax.legend(title='Injection Rate', fontsize=9)
    ax.set_title(f'{ylabel} at Injector ({INJ_WELL})', fontweight='bold')

plt.suptitle('Injection Rate Sensitivity — Injector Well Response', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── 4. Pressure Propagation Across Unique Observation Wells ───────────────
ref_run = list(runs.keys())[0]
# Build unique wells (one per grid cell) sorted by distance from injector
seen, unique_obs = set(), []
for wname, winfo in runs[ref_run]['obs_wells'].items():
    cell = (winfo['GridI'], winfo['GridJ'])
    if cell not in seen:
        seen.add(cell)
        d = math.sqrt((winfo['GridI']-51)**2 + (winfo['GridJ']-51)**2)
        unique_obs.append((wname, winfo['depth_m'], d * 0.9))
unique_obs.sort(key=lambda x: x[2])

mid_run = list(runs.keys())[2]
mid_rd  = runs[mid_run]
palette_obs = cm.coolwarm(np.linspace(0, 1, len(unique_obs)))

fig, ax = plt.subplots(figsize=(13, 5))
for i, (wname, depth, d_km) in enumerate(unique_obs):
    if wname in mid_rd['dfs']:
        df = mid_rd['dfs'][wname]
        ax.plot(df['Time_yr'], df['Pressure_bar'], color=palette_obs[i],
                linewidth=1.8, label=f'{wname}  ({d_km:.1f} km from inj.)')
try:
    ie = float(mid_rd['meta'].get('inj_end', '50').split()[0])
    ax.axvline(ie, color='gray', ls=':', lw=1.5, label=f'Injection end ({ie:.0f} yr)')
except: pass
ax.set_xlabel('Time (years)', fontsize=11)
ax.set_ylabel('Pressure (bar)', fontsize=11)
ax.set_title(f'Pressure Propagation | Scenario: {mid_rd["label"]}', fontweight='bold')
ax.legend(title='Well (dist from injector)', fontsize=8, loc='upper right', ncol=2)
ax.grid(True, ls='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5. Formation Capacity Analysis ─────────────────────────────────────────
cap_data = []
for rf, rd in runs.items():
    if not rd['active_wells']: continue
    rate_val = list(rd['active_wells'].values())[0]['rate_mtyr']
    if INJ_WELL in rd['dfs']:
        df    = rd['dfs'][INJ_WELL]
        p0    = df['Pressure_bar'].iloc[0]
        pmax  = df['Pressure_bar'].max()
        pfin  = df['Pressure_bar'].iloc[-1]
        try:    co2 = float(rd['meta'].get('co2_total','0').split()[0])
        except: co2 = 0.0
        cap_data.append({'Rate': rate_val, 'P_init': p0, 'P_peak': pmax, 'P_final': pfin, 'dP': pmax-p0, 'CO2_Mt': co2})

df_cap = pd.DataFrame(cap_data).sort_values('Rate')

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
axes[0].plot(df_cap['Rate'], df_cap['P_peak'], 'o-', color='#d62728', lw=2, markersize=8)
axes[0].set_xlabel('Injection Rate (Mt/yr)', fontsize=11)
axes[0].set_ylabel('Peak Reservoir Pressure (bar)', fontsize=11)
axes[0].set_title('Peak Pressure vs Injection Rate', fontweight='bold')
axes[0].grid(True, ls='--', alpha=0.5)

axes[1].bar(df_cap['Rate'].astype(str), df_cap['dP'],
            color=cm.plasma(np.linspace(0.1, 0.9, len(df_cap))), edgecolor='k', linewidth=0.8)
axes[1].set_xlabel('Injection Rate (Mt/yr)', fontsize=11)
axes[1].set_ylabel('ΔP = P_peak − P_initial (bar)', fontsize=11)
axes[1].set_title('Pressure Buildup ΔP', fontweight='bold')
axes[1].grid(True, ls='--', alpha=0.5, axis='y')

axes[2].plot(df_cap['Rate'], df_cap['P_final'], 's-', color='#1f77b4', lw=2, markersize=8)
axes[2].axhline(df_cap['P_init'].mean(), color='k', ls='--', label='Initial P')
axes[2].set_xlabel('Injection Rate (Mt/yr)', fontsize=11)
axes[2].set_ylabel('Pressure at Year 1000 (bar)', fontsize=11)
axes[2].set_title('Post-Injection Pressure at Year 1000', fontweight='bold')
axes[2].legend(fontsize=9)
axes[2].grid(True, ls='--', alpha=0.5)

plt.suptitle('Formation Pressure Capacity — Injector Well (31/05/07)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()
print('Summary:')
print(df_cap[['Rate','CO2_Mt','P_init','P_peak','dP','P_final']].rename(
    columns={'Rate':'Rate (Mt/yr)','CO2_Mt':'Total CO₂ (Mt)','P_init':'P_init (bar)',
             'P_peak':'P_peak (bar)','dP':'ΔP (bar)','P_final':'P_final@1000yr (bar)'}).to_string(index=False))

In [ ]:
# ── 6. Scenario × Well Heatmap ─────────────────────────────────────────────
unique_wells_hm = [(w, d) for w, d, _ in unique_obs]
rate_labels_hm  = [rd['label'] for rd in runs.values()]
n_w, n_r = len(unique_wells_hm), len(runs)

hm_pp = np.full((n_w, n_r), np.nan)
hm_sp = np.full((n_w, n_r), np.nan)
hm_sf = np.full((n_w, n_r), np.nan)

for j, (_, rd) in enumerate(runs.items()):
    for i, (wname, _) in enumerate(unique_wells_hm):
        if wname in rd['dfs']:
            df = rd['dfs'][wname]
            hm_pp[i,j] = df['Pressure_bar'].max()
            hm_sp[i,j] = df['CO2_Saturation'].max()
            hm_sf[i,j] = df['CO2_Saturation'].iloc[-1]

y_labels = [f'{w} ({d:.0f} m)' for w, d in unique_wells_hm]
fig, axes = plt.subplots(1, 3, figsize=(18, max(5, n_w*0.7)))
for ax, data, title, cmap in zip(axes,
    [hm_pp, hm_sp, hm_sf],
    ['Peak Pressure (bar)', 'Peak CO₂ Saturation', 'Final CO₂ Sat @ Yr 1000'],
    ['coolwarm', 'YlOrRd', 'YlOrRd']):
    im = ax.imshow(data, aspect='auto', cmap=cmap)
    ax.set_xticks(range(n_r))
    ax.set_xticklabels(rate_labels_hm, rotation=35, ha='right', fontsize=8)
    ax.set_yticks(range(n_w))
    ax.set_yticklabels(y_labels, fontsize=8)
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.set_xlabel('Injection Rate', fontsize=9)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    for ii in range(n_w):
        for jj in range(n_r):
            if not np.isnan(data[ii,jj]):
                ax.text(jj, ii, f'{data[ii,jj]:.2f}', ha='center', va='center', fontsize=6.5)
plt.suptitle('Scenario × Well Matrix (unique grid-cell wells, sorted by dist from injector)',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── 7. Pressure Recovery Time ──────────────────────────────────────────────
relax_data = []
for rf, rd in runs.items():
    if not rd['active_wells']: continue
    rate_val = list(rd['active_wells'].values())[0]['rate_mtyr']
    if INJ_WELL not in rd['dfs']: continue
    df = rd['dfs'][INJ_WELL]
    p0 = df['Pressure_bar'].iloc[0]
    pmax = df['Pressure_bar'].max()
    try:    ie = float(rd['meta'].get('inj_end','50').split()[0])
    except: ie = 50.0
    post = df[df['Time_yr'] > ie]
    rec  = post[post['Pressure_bar'] <= p0 + 5.0]
    t_rec = rec['Time_yr'].iloc[0] if not rec.empty else None
    relax_data.append({'Rate': rate_val, 'dP': pmax - p0, 'T_recover_yr': t_rec})

df_relax = pd.DataFrame(relax_data).sort_values('Rate')
colors   = ['#2ca02c' if t is not None else '#d62728' for t in df_relax['T_recover_yr']]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(df_relax['Rate'].astype(str), df_relax['T_recover_yr'].fillna(1050),
       color=colors, edgecolor='k', lw=0.8)
ax.axhline(1000, color='gray', ls='--', label='Sim end (1000 yr)')
ax.set_xlabel('Injection Rate (Mt/yr)', fontsize=11)
ax.set_ylabel('Time to Pressure Recovery (years)', fontsize=11)
ax.set_title('Formation Pressure Recovery Time\n(within 5 bar of initial, after injection ends)', fontweight='bold')
for idx_row, row in df_relax.iterrows():
    t_str = f"{row['T_recover_yr']:.0f} yr" if row['T_recover_yr'] else 'Not recovered'
    bar_i = list(df_relax['Rate']).index(row['Rate'])
    ax.text(bar_i, (row['T_recover_yr'] or 1050) + 15, t_str,
            ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, ls='--', alpha=0.4, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# ── 8. Interactive Multi-Well Explorer ────────────────────────────────────
all_unique_wells_exp = sorted(set(w for rd in runs.values() for w in rd['dfs'].keys()))
run_labels = {rf: f"{rd['label']} ({rf})" for rf, rd in runs.items()}

w_run = widgets.Dropdown(
    options=[(v,k) for k,v in run_labels.items()],
    description='Scenario:',
    style={'description_width':'initial'},
    layout=widgets.Layout(width='55%')
)
w_wells = widgets.SelectMultiple(
    options=all_unique_wells_exp, value=all_unique_wells_exp[:3],
    description='Wells (multi):',
    style={'description_width':'initial'},
    layout=widgets.Layout(width='55%', height='130px')
)
w_metric = widgets.RadioButtons(
    options=['Pressure_bar','CO2_Saturation','Both'], value='Both',
    description='Show:', style={'description_width':'initial'}
)
w_compare = widgets.Checkbox(
    value=False, description='Compare same wells across ALL scenarios',
    style={'description_width':'initial'}
)
out_exp = widgets.Output()

def update_explorer(change=None):
    with out_exp:
        clear_output(wait=True)
        sel_run   = w_run.value
        sel_wells = list(w_wells.value)
        metric    = w_metric.value
        compare   = w_compare.value
        if not sel_wells:
            print('Select at least one well.'); return

        n_rows = 2 if metric == 'Both' else 1
        fig, axes = plt.subplots(n_rows, 1, figsize=(12, 4*n_rows), sharex=True)
        if n_rows == 1: axes = [axes]
        pal = plt.cm.tab10.colors

        if compare:
            for wi, wname in enumerate(sel_wells):
                for ri, (rf, rd) in enumerate(runs.items()):
                    if wname not in rd['dfs']: continue
                    df = rd['dfs'][wname]
                    ls = ['-','--',':','-.'][ri%4]
                    lbl = f'{wname} @ {rd["label"]}'
                    if metric in ('Pressure_bar','Both'):
                        axes[0].plot(df['Time_yr'], df['Pressure_bar'], ls, color=pal[wi%10], lw=1.8, label=lbl)
                    if metric in ('CO2_Saturation','Both'):
                        axes[-1].plot(df['Time_yr'], df['CO2_Saturation'], ls, color=pal[wi%10], lw=1.8, label=lbl)
            title = 'Cross-Scenario Comparison — Selected Wells'
        else:
            rd = runs[sel_run]
            for wi, wname in enumerate(sel_wells):
                if wname not in rd['dfs']: continue
                df = rd['dfs'][wname]
                if metric in ('Pressure_bar','Both'):
                    axes[0].plot(df['Time_yr'], df['Pressure_bar'], color=pal[wi%10], lw=2, label=wname)
                if metric in ('CO2_Saturation','Both'):
                    axes[-1].plot(df['Time_yr'], df['CO2_Saturation'], color=pal[wi%10], lw=2, label=wname)
            title = f'Scenario: {run_labels[sel_run]}'

        ylabels = {'Pressure_bar': 'Pressure (bar)', 'CO2_Saturation': 'CO₂ Saturation'}
        metrics_used = ['Pressure_bar','CO2_Saturation'] if metric=='Both' else [metric]
        for ax, m in zip(axes, metrics_used):
            ax.set_ylabel(ylabels[m], fontsize=11)
            ax.grid(True, ls='--', alpha=0.5)
            ax.legend(fontsize=8, loc='upper right', ncol=2)
        axes[-1].set_xlabel('Time (years)', fontsize=11)
        plt.suptitle(title, fontsize=12, fontweight='bold')
        plt.tight_layout()
        plt.show()

for w in [w_run, w_wells, w_metric, w_compare]:
    w.observe(update_explorer, names='value')

display(widgets.VBox([
    widgets.HTML('<h3>Section 8 — Interactive Multi-Well Explorer</h3>'),
    widgets.HBox([w_run, w_metric]),
    widgets.HBox([w_wells, w_compare]),
    out_exp
]))
update_explorer()